# Quickstart

Multiplied is a library for exploring and quickly defining [combinational](https://en.wikipedia.org/wiki/Combinational_logic)
multiplication algorithms. The library also bundles built-in tools to analyse and
visualise algorithms through [Pandas](https://pandas.pydata.org/) and [Matplotlib](https://matplotlib.org/).

This guide will quickly walk through each area of Multiplied, without delving too
deeply into the details.


## Algorithms

A given combinational multiplier is defined by a sequence of "stages" which continuously
reduce an initial set of [partial products](https://en.wikipedia.org/wiki/Binary_multiplier#Binary_long_multiplication)
into a single output product.

First import the module and define the bitwidth of the algorithm:

In [1]:
import multiplied as mp

alg = mp.Algorithm(4)

Then automatically generate a basic algorithm.

In [2]:
alg.auto_resolve_stage(recursive=True) # recursive=True -- default
print(alg)


0:{

template:{

____AaAa
___aAaA_
__AaAa__
_BbBb___

__AaAaAa
__aAaA__
_BbBb___
________
}

pseudo:{

__AaAaAa
__aAaA__
_BbBb___
________
}

map:{

00 00 00 00 00 00 00 00 
00 00 00 00 00 00 00 00 
00 00 00 00 00 00 00 00 
FF FF FF FF FF FF FF FF 
}

1:{

template:{

__AaAaAa
__AaAa__
_aAaA___
________

_aAaAaAa
_AaAa___
________
________
}

pseudo:{

_aAaAaAa
_AaAa___
________
________
}

map:{

00 00 00 00 00 00 00 00 
00 00 00 00 00 00 00 00 
00 00 00 00 00 00 00 00 
00 00 00 00 00 00 00 00 
}

2:{

template:{

_aAaAaAa
_aAaA___
________
________

AaAaAaAa
________
________
________
}

pseudo:{

AaAaAaAa
________
________
________
}

map:{

00 00 00 00 00 00 00 00 
00 00 00 00 00 00 00 00 
00 00 00 00 00 00 00 00 
00 00 00 00 00 00 00 00 
}



As the output shows, each stage is made up of a [Template](guide/structures.html#Template),
pseudo [Matrix](guide/structures.html#Matrix), and a [Map](guide/structures.html#Map).

**Templates**:

- Represent arithmetic units via characters
- Resultant templates show where bits will be placed after a given stage

**Pseudo Matrix**:

- Shows the partial product matrix after reduction and maps have been applied
- Each matrix has a width two times it's height:
  - two values of x-bits can multiply to produce a 2x-bit value

**Maps**:

- 2-bit hexadecimal values define how far each bit is vertically shifted after reduction
- Positive values shift bits up
- negative values shift bits down


The algorithm can now execute using input operands to verify it works:

In [3]:
a = 15
b = 13
output = alg.exec(a, b)

for m in output.values():
    print(m)

# convert result to decimal
print(int("".join(alg.matrix.matrix[0]), 2))
print(a*b)

____1111
___0000_
__1111__
_1111___

__110011
__0110__
_1111___
________

_1010011
_1110___
________
________

11000011
________
________
________

195
195


Stage 0 represents the initial starting partial product matrix with each following
stage being reduced by [Adders](https://en.wikipedia.org/wiki/Adder_(electronics))
("units" which cover 2 rows) and [Carry Save Adders](https://en.wikipedia.org/wiki/Carry-save_adder)
("units" which cover 3 rows).

With the algorithm verified, a truth table can be generated:

In [4]:
import pandas as pd

domain_ = (1, 15) # range of possible operand values for a and b
range_ = (1, 255) # range of possible output values
scope = mp.truth_scope(domain_, range_) # generator clamps range to domain

# scope yields input tuples (a, b) to generate a Pandas DataFrame
df = mp.truth_dataframe(scope, alg)

# trim final columns containing formatted outputs for better viewing
df.iloc[:, :-4]

,a,b,output,s0_p0_b7,s0_p0_b6,s0_p0_b5,s0_p0_b4,s0_p0_b3,s0_p0_b2,s0_p0_b1,...,s3_p2_b1,s3_p2_b0,s3_p3_b7,s3_p3_b6,s3_p3_b5,s3_p3_b4,s3_p3_b3,s3_p3_b2,s3_p3_b1,s3_p3_b0
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,2,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,3,3,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,4,4,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,5,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,15,11,165,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
221,15,12,180,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
222,15,13,195,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
223,15,14,210,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


The truth table can be defined by three regions:

**Operands**:

a, b and output

**Partial Products**: s{x}_p{y}_b{z}

- s{x} - Stage x
- p{y} - Partial product y
- b{z} - Bit z

**Formatted Output**:

Stores outputs produced by a given execution of the algorithm as seen above


Finally, the generated data is ready to be visualised. Let's keep it simple and
generate a 2D heatmap:

<!--```{code-cell}

mp.df_global_heatmap()


```-->